# Fusão de Bases de Dados (Merge) - fNIRS e Fisiologia Periférica
Este script cruza as extrações neurofisiológicas (GLM Betas e p-values do córtex pré-frontal) com as métricas autonómicas sistémicas (GSR e HR) da tarefa de Decision Making. A fusão é realizada através da chave de identificação do sujeito (`ID`), preparando a base de dados final (n=54) para análise estatística inferencial.

In [2]:
import os
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. Configuração de Caminhos
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
PASTA_MNE_NIRS = os.path.join(BASE_DIR, "derivatives", "mne-nirs")

FICHEIRO_FNIRS = "dataset_mestre_fnirs_DM.csv"
FICHEIRO_FISIO = "dataset_mestre_fisio_DM.csv"

caminho_fnirs = os.path.join(PASTA_MNE_NIRS, FICHEIRO_FNIRS)
caminho_fisio = os.path.join(PASTA_MNE_NIRS, FICHEIRO_FISIO)

print("A iniciar processo de fusão (Merge)...")

try:
    # ==========================================
    # 2. Leitura dos Dados
    # ==========================================
    df_fnirs = pd.read_csv(caminho_fnirs)
    
    # O engine 'python' com sep=None deteta automaticamente se o teu ficheiro
    # está separado por vírgula (,) ou ponto-e-vírgula (;)
    df_fisio = pd.read_csv(caminho_fisio, sep=None, engine='python')
    
    # Garantir que a chave de cruzamento (ID) não tem espaços em branco acidentais
    df_fnirs['ID'] = df_fnirs['ID'].str.strip()
    df_fisio['ID'] = df_fisio['ID'].str.strip()

    # Remover colunas duplicadas que não queremos repetir (ex: Tarefa)
    if 'Tarefa' in df_fisio.columns:
        df_fisio = df_fisio.drop(columns=['Tarefa'])

    # ==========================================
    # 3. Merge (Fusão) das Matrizes
    # ==========================================
    print("A cruzar os dados usando a coluna 'ID'...")
    
    # O merge 'inner' mantém apenas os sujeitos presentes EM AMBOS os ficheiros
    df_master = pd.merge(df_fnirs, df_fisio, on='ID', how='inner')
    
    # ==========================================
    # 4. Ordenação das Colunas
    # ==========================================
    # O ID, Tarefa, Qualidade SCI e Fisiologia no início, seguidos de todos os canais HbO
    colunas_base = ['ID', 'Tarefa', 'Canais_Rejeitados_SCI']
    
    # Detetar dinamicamente as colunas de fisiologia presentes
    colunas_fisiologia = [c for c in df_fisio.columns if c != 'ID']
    
    # Detetar colunas do fNIRS
    colunas_hbo = [col for col in df_fnirs.columns if col.startswith('HbO_')]
    
    # Reordenar dataframe
    todas_colunas_ordenadas = colunas_base + colunas_fisiologia + colunas_hbo
    df_master = df_master[todas_colunas_ordenadas]

    # ==========================================
    # 5. Exportação Final
    # ==========================================
    caminho_exportacao = os.path.join(PASTA_MNE_NIRS, "dataset_FINAL_SPA_fNIRS_DecisionMaking.csv")
    df_master.to_csv(caminho_exportacao, index=False)
    
    print("\n" + "="*50)
    print("FUSÃO CONCLUÍDA COM SUCESSO!")
    print("="*50)
    print(f"-> Número de sujeitos integrados: {df_master.shape[0]}")
    print(f"-> Total de variáveis na tabela: {df_master.shape[1]}")
    print(f"-> Ficheiro guardado em:\n{caminho_exportacao}\n")
    
    # Mostrar um pequeno resumo para confirmares se a fisiologia alinhou bem
    print("Amostra das primeiras 2 linhas e 7 colunas iniciais:")
    display(df_master.iloc[:2, :7])

except Exception as e:
    print(f"ERRO durante a leitura/fusão: {e}")

A iniciar processo de fusão (Merge)...
A cruzar os dados usando a coluna 'ID'...

FUSÃO CONCLUÍDA COM SUCESSO!
-> Número de sujeitos integrados: 54
-> Total de variáveis na tabela: 47
-> Ficheiro guardado em:
C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI\derivatives\mne-nirs\dataset_FINAL_SPA_fNIRS_DecisionMaking.csv

Amostra das primeiras 2 linhas e 7 colunas iniciais:


,ID,Tarefa,Canais_Rejeitados_SCI,GSR_Media,GSR_Amplitude_Max,HR_Media,HR_Bradicardia_Max
0,sub-001,Decision_Making,18,-0.607718,0.237509,-1.266939,-3.060944
1,sub-002,Decision_Making,16,-0.073031,0.478657,-0.407013,-1.563385
